In [ ]:
# Dedup API Performance Test - 1M records with retry + latency tracking
# Everything in a single cell

import requests
import traceback
import time
import json
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import StructType, StructField, DoubleType, StringType, IntegerType

# ---------- Widgets / config ----------
dbutils.widgets.dropdown("endpoint", defaultValue="staging", choices=["staging", "prod", "old_prod"])
dbutils.widgets.text("record_count", defaultValue="1000000")

endpoint_lookups = {
    "staging":  "https://pcis-model-service-staging.apps.k8s.uscis.dhs.gov/api/v1/model/dedup/predict",
    "prod":     "https://pcis-model-service-pr.apps.k8s.uscis.dhs.gov/api/v1/model/dedup/predict",
    "old_prod": "https://modeling-service-prod.apps.k8s.uscis.dhs.gov/api/v1/model/dedup/predict",
}
partition_size_lookups = {"staging": 200, "prod": 200, "old_prod": 200}

model_endpoint = endpoint_lookups[dbutils.widgets.get("endpoint")]
partition_size = partition_size_lookups[dbutils.widgets.get("endpoint")]
record_count = int(dbutils.widgets.get("record_count"))

BEARER_TOKEN = dbutils.secrets.get(scope="pcis_dhs", key="dedup_bearer_token")
headers = {"Authorization": f"Bearer {BEARER_TOKEN}"}

RETRY_STATUS_CODES = {502, 503, 504}
MAX_RETRIES = 3
BACKOFF_BASE = 0.5

print(f"model_endpoint: {model_endpoint}")
print(f"partition_size: {partition_size}")
print(f"record_count:   {record_count:,}")

# ---------- Build 1M-row input by replicating 50K source ----------
source_df = (
    spark.table("pcis_data_science.small_static_test_set_regression_testing")
    .withColumnRenamed("dob1", "dateOfBirth1")
    .withColumnRenamed("cob1", "countryOfBirth1")
    .withColumnRenamed("coc1", "countryOfCitizenship1")
    .withColumnRenamed("dob2", "dateOfBirth2")
    .withColumnRenamed("cob2", "countryOfBirth2")
    .withColumnRenamed("coc2", "countryOfCitizenship2")
)

source_count = source_df.count()
multiplier = int(record_count // source_count) + 1
print(f"source: {source_count:,}, multiplier: {multiplier}x")

dedup_df = (
    source_df
    .withColumn("_rep", F.explode(F.array_repeat(F.lit(1), multiplier)))
    .drop("_rep")
    .limit(record_count)
    .repartition(partition_size)
)
print(f"dedup_df rows: {dedup_df.count():,}")

# ---------- UDF with retry + latency tracking ----------
schema = StructType([
    StructField("request", StringType()),
    StructField("response_status_code", StringType()),
    StructField("response_text", StringType()),
    StructField("score", DoubleType()),
    StructField("error", StringType()),
    StructField("package_version", StringType()),
    StructField("attempts", IntegerType()),
    StructField("latency_ms", DoubleType()),
])


@pandas_udf(schema)
def dedup_udf(
    partySourceId2: pd.Series, firstName2: pd.Series, middleName2: pd.Series, lastName2: pd.Series, fullName2: pd.Series,
    dateOfBirth2: pd.Series, aNumber2: pd.Series, countryOfBirth2: pd.Series, countryOfCitizenship2: pd.Series,
    ssn2: pd.Series, fin2: pd.Series, motherName2: pd.Series, fatherName2: pd.Series,
    partySourceId1: pd.Series, firstName1: pd.Series, middleName1: pd.Series, lastName1: pd.Series, fullName1: pd.Series,
    dateOfBirth1: pd.Series, aNumber1: pd.Series, countryOfBirth1: pd.Series, countryOfCitizenship1: pd.Series,
    ssn1: pd.Series, fin1: pd.Series, motherName1: pd.Series, fatherName1: pd.Series,
) -> pd.DataFrame:

    out = {
        "request": [],
        "response_status_code": [],
        "response_text": [],
        "score": [],
        "error": [],
        "package_version": [],
        "attempts": [],
        "latency_ms": [],
    }

    for (psid2, fn2, mn2, ln2, fln2, dob2, an2, cob2, coc2, ssn_2, fin_2, mom2, dad2,
         psid1, fn1, mn1, ln1, fln1, dob1, an1, cob1, coc1, ssn_1, fin_1, mom1, dad1) in zip(
        partySourceId2, firstName2, middleName2, lastName2, fullName2, dateOfBirth2, aNumber2,
        countryOfBirth2, countryOfCitizenship2, ssn2, fin2, motherName2, fatherName2,
        partySourceId1, firstName1, middleName1, lastName1, fullName1, dateOfBirth1, aNumber1,
        countryOfBirth1, countryOfCitizenship1, ssn1, fin1, motherName1, fatherName1,
    ):
        payload = {
            "targetService": "dedup",
            "candidates": [{
                "candidateId": psid2,
                "firstName": fn2,
                "middleName": mn2,
                "lastName": ln2,
                "fullName": fln2,
                "dateOfBirth": dob2,
                "aNumber": an2,
                "countryOfBirth": cob2,
                "countryOfCitizenship": coc2,
                "ssn": ssn_2,
                "fin": fin_2,
                "motherName": mom2,
                "fatherName": dad2,
            }],
            "declared": {
                "partySourceId": psid1,
                "firstName": fn1,
                "middleName": mn1,
                "lastName": ln1,
                "fullName": fln1,
                "dateOfBirth": dob1,
                "aNumber": an1,
                "countryOfBirth": cob1,
                "countryOfCitizenship": coc1,
                "ssn": ssn_1,
                "fin": fin_1,
                "motherName": mom1,
                "fatherName": dad1,
            },
        }

        response_status_code, response_text, score, package_version = None, None, None, None
        error = "None"
        attempt_count = 0

        start = time.perf_counter()
        for attempt in range(MAX_RETRIES + 1):
            attempt_count = attempt + 1
            try:
                dedup_response = requests.post(url=model_endpoint, headers=headers, json=payload, verify=False, timeout=30)
                response_status_code = dedup_response.status_code
                response_text = dedup_response.text

                if response_status_code in RETRY_STATUS_CODES and attempt < MAX_RETRIES:
                    time.sleep(BACKOFF_BASE * (2 ** attempt))
                    continue

                response_json = dedup_response.json()
                if "candidates" in response_json:
                    package_version = response_json.get("packageVersion")
                    try:
                        score = response_json["candidates"][0]["score"]
                    except (IndexError, KeyError):
                        score = None
                break

            except (requests.ConnectionError, requests.Timeout) as e:
                if attempt < MAX_RETRIES:
                    time.sleep(BACKOFF_BASE * (2 ** attempt))
                    continue
                error = f"{type(e).__name__}: {e}"
                break

            except Exception:
                error = traceback.format_exc()
                break

        latency_ms = (time.perf_counter() - start) * 1000.0

        out["request"].append(json.dumps(payload))
        out["response_status_code"].append(str(response_status_code) if response_status_code is not None else "None")
        out["response_text"].append(response_text)
        out["score"].append(score)
        out["error"].append(error)
        out["package_version"].append(package_version)
        out["attempts"].append(attempt_count)
        out["latency_ms"].append(latency_ms)

    return pd.DataFrame(out)


# ---------- Run + write ----------
run_start = time.perf_counter()

initial_df = dedup_df.select(
    "*",
    dedup_udf(
        "partySourceId2", "firstName2", "middleName2", "lastName2", "fullName2",
        "dateOfBirth2", "aNumber2", "countryOfBirth2", "countryOfCitizenship2",
        "ssn2", "fin2", "motherName2", "fatherName2",
        "partySourceId1", "firstName1", "middleName1", "lastName1", "fullName1",
        "dateOfBirth1", "aNumber1", "countryOfBirth1", "countryOfCitizenship1",
        "ssn1", "fin1", "motherName1", "fatherName1",
    ).alias("dedup_response")
)

df = (
    initial_df
    .withColumn("score", F.col("dedup_response.score"))
    .withColumn("response_status_code", F.col("dedup_response.response_status_code"))
    .withColumn("error", F.col("dedup_response.error"))
    .withColumn("request", F.col("dedup_response.request"))
    .withColumn("attempts", F.col("dedup_response.attempts"))
    .withColumn("latency_ms", F.col("dedup_response.latency_ms"))
    .withColumn("run_at", F.current_timestamp())
    .withColumn("endpoint", F.lit(model_endpoint))
    .withColumn("package_version", F.col("dedup_response.package_version"))
    .withColumn("response_text", F.col("dedup_response.response_text"))
    .drop("dedup_response")
)

df.write.mode("overwrite").saveAsTable("pcis_data_science.dedup_perf_test_results")

elapsed = time.perf_counter() - run_start
print(f"Wall clock: {elapsed:.1f}s  ({record_count / elapsed:.1f} req/s)")